## Task 15: Enterprise LLM Gateway with Rate-Limiting & Fallback
**Requires (production):** FastAPI, a running Redis instance, Docker, and Prometheus/Grafana for metrics. Since there's no Redis server or Docker in this sandbox, the core gateway *logic* — token-bucket rate limiting and automatic model fallback — is implemented and tested below using a plain in-memory store standing in for Redis, followed by the full production FastAPI app for reference.

In [1]:
!pip install fastapi uvicorn redis prometheus-client requests -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 560.6/560.6 kB 8.6 MB/s eta 0:00:00


In [2]:

# Install Redis server
!apt-get update -qq
!apt-get install -y redis-server -qq

# Start Redis in the background
!redis-server --daemonize yes

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libjemalloc2:amd64.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../0-libjemalloc2_5.2.1-4ubuntu1_amd64.deb ...
Unpacking libjemalloc2:amd64 (5.2.1-4ubuntu1) ...
Selecting previously unselected package liblua5.1-0:amd64.
Preparing to unpack .../1-liblua5.1-0_5.1.5-8.1build4_amd64.deb ...
Unpacking liblua5.1-0:amd64 (5.1.5-8.1build4) ...
Selecting previously unselected package liblzf1:amd64.
Preparing to unpack .../2-liblzf1_3.6-3_amd64.deb ...
Unpacking liblzf1:amd64 (3.6-3) ...
Selecting previously unselected package lua-bitop:amd64.
Preparing to unpack .../3-lua-bitop_1.0.2-5_amd64.deb ...
Unpacking lua-bitop:amd64 (1.0.2-5) ...
Selecting previously unselected package lua-cjson:amd64.
Preparing to unpack .../4-

In [3]:

import time

class InMemoryRateLimiter:   # stand-in for Redis-backed token bucket
    def __init__(self, capacity, refill_rate):
        self.capacity = capacity
        self.refill_rate = refill_rate    # tokens per second
        self.tokens = capacity
        self.last_refill = time.time()

    def allow(self):
        now = time.time()
        elapsed = now - self.last_refill
        self.tokens = min(self.capacity, self.tokens + elapsed * self.refill_rate)
        self.last_refill = now
        if self.tokens >= 1:
            self.tokens -= 1
            return True
        return False

class LLMGateway:
    def __init__(self, primary, fallback, capacity=5, refill_rate=1):
        self.primary = primary
        self.fallback = fallback
        self.limiter = InMemoryRateLimiter(capacity, refill_rate)
        self.latencies = []

    def call_model(self, model_name, prompt):
        # simulated backend call
        if model_name == "primary" and prompt == "trigger_failure":
            raise ConnectionError("primary model down")
        return f"[{model_name}] response to: {prompt}"

    def route(self, prompt):
        start = time.time()
        if not self.limiter.allow():
            return {"status": "rate_limited"}
        try:
            result = self.call_model("primary", prompt)
            status = "ok_primary"
        except ConnectionError:
            result = self.call_model("fallback", prompt)
            status = "ok_fallback"
        self.latencies.append(time.time() - start)
        return {"status": status, "result": result}

gateway = LLMGateway(primary="gpt-4", fallback="gpt-3.5", capacity=3, refill_rate=0.5)

requests_to_simulate = ["hello", "how are you", "trigger_failure", "another one", "and another"]
for req in requests_to_simulate:
    print(gateway.route(req))

print(f"\nAvg latency: {sum(gateway.latencies)/len(gateway.latencies)*1000:.3f} ms")


{'status': 'ok_primary', 'result': '[primary] response to: hello'}
{'status': 'ok_primary', 'result': '[primary] response to: how are you'}
{'status': 'ok_fallback', 'result': '[fallback] response to: trigger_failure'}
{'status': 'rate_limited'}
{'status': 'rate_limited'}

Avg latency: 0.007 ms


In [4]:
from fastapi import FastAPI, HTTPException
import redis
import time
from prometheus_client import Counter, Histogram, make_asgi_app, REGISTRY

# ---------- Metrics (registered only once) ----------

if "gateway_requests_total" not in REGISTRY._names_to_collectors:
    REQUEST_COUNT = Counter(
        "gateway_requests_total",
        "Total requests",
        ["status"],  # ok_primary, ok_fallback, rate_limited
    )
else:
    REQUEST_COUNT = REGISTRY._names_to_collectors["gateway_requests_total"]

if "gateway_latency_seconds" not in REGISTRY._names_to_collectors:
    LATENCY = Histogram(
        "gateway_latency_seconds",
        "Request latency",
    )
else:
    LATENCY = REGISTRY._names_to_collectors["gateway_latency_seconds"]

# ---------- FastAPI app ----------

app = FastAPI()

# Redis running locally in Colab
r = redis.Redis(host="localhost", port=6379, db=0)

# Expose Prometheus metrics at /metrics
app.mount("/metrics", make_asgi_app())


def token_bucket_allow(
    key: str,
    capacity: int = 100,
    refill_rate: float = 10.0,
    redis_client: redis.Redis = r,
) -> bool:
    now = time.time()

    pipe = redis_client.pipeline()
    pipe.hget(key, "tokens")
    pipe.hget(key, "last_update")
    result = pipe.execute()
    tokens_raw, last_update_raw = result

    if tokens_raw is None:
        tokens = float(capacity)
        last_update = now
    else:
        tokens = float(tokens_raw)
        last_update = float(last_update_raw)

    elapsed = now - last_update
    tokens = min(capacity, tokens + elapsed * refill_rate)

    if tokens < 1.0:
        pipe = redis_client.pipeline()
        pipe.hset(key, mapping={"tokens": str(tokens), "last_update": str(now)})
        pipe.expire(key, 3600)
        pipe.execute()
        return False

    tokens -= 1.0

    pipe = redis_client.pipeline()
    pipe.hset(key, mapping={"tokens": str(tokens), "last_update": str(now)})
    pipe.expire(key, 3600)
    pipe.execute()
    return True


def call_primary_llm(prompt: str) -> str:
    if prompt == "trigger_failure":
        raise RuntimeError("primary model down")
    return f"[primary] response to: {prompt}"


def call_fallback_llm(prompt: str) -> str:
    return f"[fallback] response to: {prompt}"


@app.post("/v1/complete")
async def complete(prompt: str):
    with LATENCY.time():
        client_key = "gateway:client_1"

        if not token_bucket_allow(client_key, capacity=100, refill_rate=10.0):
            REQUEST_COUNT.labels(status="rate_limited").inc()
            raise HTTPException(status_code=429, detail="Rate limit exceeded")

        try:
            result = call_primary_llm(prompt)
            REQUEST_COUNT.labels(status="ok_primary").inc()
        except Exception:
            result = call_fallback_llm(prompt)
            REQUEST_COUNT.labels(status="ok_fallback").inc()

        return {"result": result}

In [5]:
import uvicorn
import threading

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

# Start server in a background thread
threading.Thread(target=run_server, daemon=True).start()

In [6]:
import requests

base = "http://localhost:8000"

# 1) Normal request -> primary
resp = requests.post(f"{base}/v1/complete", params={"prompt": "hello"})
print("Request 1:", resp.json())

# 2) Trigger failure -> fallback
resp = requests.post(f"{base}/v1/complete", params={"prompt": "trigger_failure"})
print("Request 2:", resp.json())

# 3) Another normal request
resp = requests.post(f"{base}/v1/complete", params={"prompt": "how are you?"})
print("Request 3:", resp.json())

INFO:     127.0.0.1:39778 - "POST /v1/complete?prompt=hello HTTP/1.1" 200 OK
Request 1: {'result': '[primary] response to: hello'}
INFO:     127.0.0.1:39780 - "POST /v1/complete?prompt=trigger_failure HTTP/1.1" 200 OK
Request 2: {'result': '[fallback] response to: trigger_failure'}
INFO:     127.0.0.1:39784 - "POST /v1/complete?prompt=how+are+you%3F HTTP/1.1" 200 OK
Request 3: {'result': '[primary] response to: how are you?'}


In [7]:
import requests

resp = requests.get("http://localhost:8000/metrics")
print("=== Gateway Metrics ===")
for line in resp.text.split("\n"):
    if "gateway_" in line and not line.startswith("#"):
        print(line)

INFO:     127.0.0.1:39796 - "GET /metrics HTTP/1.1" 307 Temporary Redirect
INFO:     127.0.0.1:39796 - "GET /metrics/ HTTP/1.1" 200 OK
=== Gateway Metrics ===
gateway_requests_total{status="ok_primary"} 2.0
gateway_requests_total{status="ok_fallback"} 1.0
gateway_requests_created{status="ok_primary"} 1.7862593431069317e+09
gateway_requests_created{status="ok_fallback"} 1.7862593431179926e+09
gateway_latency_seconds_bucket{le="0.005"} 3.0
gateway_latency_seconds_bucket{le="0.01"} 3.0
gateway_latency_seconds_bucket{le="0.025"} 3.0
gateway_latency_seconds_bucket{le="0.05"} 3.0
gateway_latency_seconds_bucket{le="0.075"} 3.0
gateway_latency_seconds_bucket{le="0.1"} 3.0
gateway_latency_seconds_bucket{le="0.25"} 3.0
gateway_latency_seconds_bucket{le="0.5"} 3.0
gateway_latency_seconds_bucket{le="0.75"} 3.0
gateway_latency_seconds_bucket{le="1.0"} 3.0
gateway_latency_seconds_bucket{le="2.5"} 3.0
gateway_latency_seconds_bucket{le="5.0"} 3.0
gateway_latency_seconds_bucket{le="7.5"} 3.0
gateway_la

In [8]:
import requests

base = "http://localhost:8000"

print("=== Testing Rate Limiting ===")
for i in range(15):
    resp = requests.post(f"{base}/v1/complete", params={"prompt": f"request {i}"})
    print(f"Request {i}: {resp.status_code} -> {resp.json()}")

=== Testing Rate Limiting ===
INFO:     127.0.0.1:39802 - "POST /v1/complete?prompt=request+0 HTTP/1.1" 200 OK
Request 0: 200 -> {'result': '[primary] response to: request 0'}
INFO:     127.0.0.1:39816 - "POST /v1/complete?prompt=request+1 HTTP/1.1" 200 OK
Request 1: 200 -> {'result': '[primary] response to: request 1'}
INFO:     127.0.0.1:39822 - "POST /v1/complete?prompt=request+2 HTTP/1.1" 200 OK
Request 2: 200 -> {'result': '[primary] response to: request 2'}
INFO:     127.0.0.1:39830 - "POST /v1/complete?prompt=request+3 HTTP/1.1" 200 OK
Request 3: 200 -> {'result': '[primary] response to: request 3'}
INFO:     127.0.0.1:39834 - "POST /v1/complete?prompt=request+4 HTTP/1.1" 200 OK
Request 4: 200 -> {'result': '[primary] response to: request 4'}
INFO:     127.0.0.1:39842 - "POST /v1/complete?prompt=request+5 HTTP/1.1" 200 OK
Request 5: 200 -> {'result': '[primary] response to: request 5'}
INFO:     127.0.0.1:39846 - "POST /v1/complete?prompt=request+6 HTTP/1.1" 200 OK
Request 6: 20